# Project 6 — Optimize a CUDA Kernel (naive -> coalesced -> tiled -> tree-reduce)
```
ladder:  stride (bad) -> contiguous/coalesced -> blocked/shared-mem -> warp/tree reduction
measure: GB/s per rung, speedup vs naive   ||   tools: nvidia-smi / ncu / nsys (auto-skip without them)
```
**Goal:** prove with numbers that *how* you touch memory decides kernel speed — same flops, up to 10x different runtime.
**Learning objectives:** explain coalescing (adjacent threads, adjacent bytes); explain tiling as a shared-memory/cache strategy; implement a tree reduction and state why it beats a serial accumulator; profile with nvidia-smi and describe what ncu/nsys would add.
CPU execution: every rung has a CPU analog that runs here (strided vs contiguous, naive vs blocked loops, serial vs tree reduce); GPU + profiler rows run only with CUDA/driver tools (rerun on DGX Spark).


In [ ]:
import os  # need os to create the results dir
import random  # need random to seed python rng
import time  # need time for perf_counter timing
import csv  # need csv to write the optimization table
import shutil  # need shutil to probe for nvidia-smi / ncu / nsys / nvcc
import subprocess  # need subprocess to snapshot nvidia-smi when present
import numpy as np  # need numpy for contiguous/strided arrays + blocked math
import matplotlib.pyplot as plt  # need matplotlib for speedup + bandwidth charts
import torch  # need torch for torch-side timings + optional gpu rungs
SEED = 42  # fixed seed so all experiment data is reproducible
random.seed(SEED)  # seed python rng
np.random.seed(SEED)  # seed numpy rng
torch.manual_seed(SEED)  # seed torch rng
os.makedirs('./results', exist_ok=True)  # create results folder (no error if exists)
HAS_CUDA = torch.cuda.is_available()  # true only with an nvidia gpu
HAS_SMI = shutil.which('nvidia-smi') is not None  # true when the nvidia driver tool exists
HAS_NCU = shutil.which('ncu') is not None  # true when nsight-compute cli is installed
HAS_NSYS = shutil.which('nsys') is not None  # true when nsight-systems cli is installed
HAS_NVCC = shutil.which('nvcc') is not None  # true when the cuda toolkit compiler is on path
print('torch', torch.__version__, '| cuda?', HAS_CUDA)  # log core env
print(f'nvidia-smi={HAS_SMI} ncu={HAS_NCU} nsys={HAS_NSYS} nvcc={HAS_NVCC}')  # log profiler toolchain flags
def sync_if_cuda():  # block until queued gpu kernels finish (correct timing)
    if HAS_CUDA:  # sync only when cuda exists
        torch.cuda.synchronize()  # wait for async gpu work
def time_op(fn, warmup=2, repeat=5):  # median timer for FAST ops (numpy/torch kernels)
    for _ in range(warmup):  # warmup iters warm caches + cuda autotune
        fn()  # untimed warmup run
    sync_if_cuda()  # sync before the timed region
    ts = []  # per-iter times go here
    for _ in range(repeat):  # timed repetitions
        t0 = time.perf_counter()  # start timer
        fn()  # run once
        sync_if_cuda()  # include async gpu work in the timer
        ts.append(time.perf_counter() - t0)  # record seconds
    return float(np.median(ts))  # median resists outliers
def ms_once(fn):  # single-shot timer for SLOW ops (pure-python loops get one shot)
    t0 = time.perf_counter()  # start timer
    out = fn()  # run once (no repeat: minutes-long otherwise)
    return (time.perf_counter() - t0), out  # return (seconds, result) pair
print('harness ready: time_op (fast, median) + ms_once (slow, single-shot)')  # confirm timers

In [ ]:
R = 2048  # experiment-a size: 2048x2048 fp32 = 16mb (streams from dram, not cache)
big = np.random.default_rng(SEED).standard_normal((R, R), dtype=np.float32)  # one big row-major matrix
print('array', big.shape, 'C-contiguous:', big.flags['C_CONTIGUOUS'], 'nbytes =', big.nbytes // 2**20, 'MB')  # confirm layout + size
contig = big[:, :256]  # RUNG 1 (good): contiguous chunk view, 524k elements read sequentially
strided = big[:, ::8]  # RUNG 0 (bad): strided chunk view, SAME 524k elements, one float per 32-byte span
print('equal elements:', contig.size == strided.size, '| bytes moved per op:', contig.nbytes * 2 // 2**10, 'KB (read+write)')  # fair-fight check
def stream_square(M):  # streaming kernel analog: out = M*M (one read pass + one write pass)
    return np.square(M)  # vectorized elementwise square (identical math for both views)
c1 = stream_square(contig)  # warmup + runnable check on the good view
s1 = stream_square(strided)  # warmup on the bad view
t_good = time_op(lambda: stream_square(contig))  # time contiguous rung (median of 5)
t_bad = time_op(lambda: stream_square(strided))  # time strided rung (median of 5)
touched = contig.nbytes * 2  # bytes moved per op: full read + full write of the chunk
gbs_good = touched / t_good / 1e9  # achieved bandwidth, good pattern
gbs_bad = touched / t_bad / 1e9  # achieved bandwidth, bad pattern (same bytes, more cache lines)
print(f'contiguous chunk: {t_good*1e3:.2f}ms ({gbs_good:.1f} GB/s)')  # good-rung line
print(f'strided chunk:    {t_bad*1e3:.2f}ms ({gbs_bad:.1f} GB/s) -> {t_bad / t_good:.1f}x slower, SAME elements+flops')  # bad-rung line
print('why: strided touches ~8x the cache lines for the same floats (one useful 4B per 32B span)')  # mechanism line
print('cuda lesson: adjacent threads must touch adjacent bytes (coalescing); strided turns one 128B transaction into many')  # takeaway
ct = torch.from_numpy(contig)  # torch view of the contiguous chunk (shared memory, no copy)
st = torch.from_numpy(strided)  # torch view of the strided chunk (torch supports strided views)
t_tgood = time_op(lambda: torch.square(ct))  # torch contiguous timing
t_tbad = time_op(lambda: torch.square(st))  # torch strided timing
print(f'torch contiguous: {t_tgood*1e3:.2f}ms | torch strided: {t_tbad*1e3:.2f}ms ({t_tbad / t_tgood:.1f}x)')  # torch confirms the gap
assert np.allclose(c1, torch.square(ct).numpy(), atol=1e-6)  # numpy and torch agree (contiguous view)
assert np.allclose(s1, torch.square(st).numpy(), atol=1e-6)  # numpy and torch agree (strided view)
if HAS_CUDA:  # gpu rungs (dgx spark): same fair fight in vram where uncoalesced hurts more
    cg = torch.from_numpy(contig).to('cuda')  # contiguous chunk on gpu (copy once, outside timers)
    sg = torch.from_numpy(strided).to('cuda')  # strided chunk on gpu (copy once, outside timers)
    tg_good = time_op(lambda: torch.square(cg))  # coalesced gpu timing
    tg_bad = time_op(lambda: torch.square(sg))  # uncoalesced gpu timing
    print(f'gpu coalesced: {tg_good*1e3:.2f}ms | gpu strided: {tg_bad*1e3:.2f}ms ({tg_bad / tg_good:.1f}x)')  # gpu gap line
    GPU_COAL = (tg_good, tg_bad)  # stash gpu times for the csv
else:  # cpu-only box: record the skip
    print('gpu coalescing rungs SKIPPED (no cuda); expect a LARGER gap on hbm (more lanes to starve)')  # guidance
    GPU_COAL = None  # marker for the csv writer
print('experiment A done: layout is performance (same math, different speed)')  # section close

In [ ]:
M = 96  # experiment-b matrix size: 96^3 = 884k macs (python loops stay in seconds, not minutes)
A = np.random.default_rng(SEED + 1).standard_normal((M, M))  # plain float64 test matrix a
B = np.random.default_rng(SEED + 2).standard_normal((M, M))  # plain float64 test matrix b
C_ref = A @ B  # blas oracle both python rungs must match
def matmul_naive(A, B):  # RUNG 0 (bad): ijk triple loop, no reuse (every mac reloads from dram/cache)
    n = A.shape[0]  # read matrix size once
    C = np.zeros((n, n))  # allocate output accumulator
    for i in range(n):  # loop rows of A / C
        for j in range(n):  # loop columns of B / C
            s = 0.0  # per-element accumulator (a register on gpu)
            for k in range(n):  # inner-product loop (streams a full row + column per element)
                s += A[i, k] * B[k, j]  # one multiply-add from two loads (zero reuse of B column)
            C[i, j] = s  # write the finished element
    return C  # return the product
def matmul_blocked_py(A, B, T=32):  # RUNG 1: same math, tiled loop order (cache/shared-mem structure)
    n = A.shape[0]  # read matrix size once
    C = np.zeros((n, n))  # allocate output accumulator
    for ii in range(0, n, T):  # loop over row-tiles
        for jj in range(0, n, T):  # loop over column-tiles
            for kk in range(0, n, T):  # loop over reduction-tiles (the blocking axis)
                for i in range(ii, min(ii + T, n)):  # rows inside this tile (hot in cache now)
                    for j in range(jj, min(jj + T, n)):  # cols inside this tile
                        s = C[i, j]  # reload the running accumulator once per tile-visit
                        for k in range(kk, min(kk + T, n)):  # short inner loop over the resident tile
                            s += A[i, k] * B[k, j]  # macs hit cache, not dram (the whole point)
                        C[i, j] = s  # write back the updated accumulator
    return C  # return the product
t_naive, Cn = ms_once(lambda: matmul_naive(A, B))  # time the naive rung once (slow!)
assert np.allclose(Cn, C_ref, atol=1e-6)  # naive must still be CORRECT (optimization never changes answers)
print(f'naive ijk {M}x{M}: {t_naive:.2f}s')  # naive timing line
t_blockpy, Cb = ms_once(lambda: matmul_blocked_py(A, B, T=32))  # time the tiled python rung once
assert np.allclose(Cb, C_ref, atol=1e-6)  # tiled must match blas too
print(f'blocked-python T=32: {t_blockpy:.2f}s ({t_naive / t_blockpy:.2f}x vs naive)')  # tie-or-win line
print('honest reading: 96x96 (73KB) fits in L1/L2, so there is NO memory cliff here -> blocking cannot win (the tie IS the roofline prediction)')  # the lesson
M2 = 384  # experiment-b realistic size: 384^3 = 113m flops (blas territory; python-naive would take ~10s)
A2 = np.random.default_rng(SEED + 1).standard_normal((M2, M2))  # bigger test matrix a
B2 = np.random.default_rng(SEED + 2).standard_normal((M2, M2))  # bigger test matrix b
C2_ref = A2 @ B2  # blas oracle for the realistic size
def matmul_blocked_np(A, B, T=128):  # RUNG 2: tile-loop STRUCTURE, but each tile multiplies via blas (the gpu recipe)
    n = A.shape[0]  # read matrix size once
    C = np.zeros((n, n))  # allocate output accumulator
    for ii in range(0, n, T):  # row-tiles (one thread-block's C-tile on gpu)
        for jj in range(0, n, T):  # column-tiles
            for kk in range(0, n, T):  # reduction-tiles (the shared-mem stage-in axis)
                C[ii:ii+T, jj:jj+T] += A[ii:ii+T, kk:kk+T] @ B[kk:kk+T, jj:jj+T]  # tile mac via fast math (tensor-core analog)
    return C  # return the product
t_blas = time_op(lambda: A2 @ B2, repeat=10)  # blas ceiling at realistic size (fast: more reps ok)
gflops_blas = 2 * M2 ** 3 / t_blas / 1e9  # ceiling throughput in gflop/s
print(f'blas {M2}x{M2}: {t_blas*1e3:.2f}ms ({gflops_blas:.1f} GFLOP/s, ceiling)')  # ceiling line
tiled = {}  # tile-size -> seconds map for the what-if sweep
for T in [96, 192]:  # sweep tile sizes (gpu analog: tuning tiles against shared-mem capacity)
    Ctb = matmul_blocked_np(A2, B2, T=T)  # run once for correctness (not timed: includes cold allocation)
    assert np.allclose(Ctb, C2_ref, atol=1e-6)  # tiled-numpy must match blas
    t_tb = time_op(lambda T=T: matmul_blocked_np(A2, B2, T=T), repeat=5)  # median of 5 (fast rung: repeats kill noise)
    tiled[T] = t_tb  # record median seconds for this tile size
    print(f'blocked-numpy T={T:>3d}: {t_tb*1e3:.2f}ms ({2 * M2**3 / t_tb / 1e9:.1f} GFLOP/s; blas {t_blas*1e3:.2f}ms)')  # speed line per tile
print('cuda lesson: __shared__ tiles = the same blocking in 96KB of sm-local sram; tensor cores = the fast tile math')  # takeaway
print('experiment B done: the tile STRUCTURE runs at ~blas speed; gpus add fast tile math (tensor cores) in shared memory')  # section close

In [ ]:
N = 4096 * 1024  # experiment-c length: 4194304 fp32 = 16mb streaming read (divisible by tree block 1024)
x = np.random.default_rng(SEED + 3).standard_normal(N, dtype=np.float32)  # test vector to sum
xt = torch.from_numpy(x)  # torch cpu view sharing memory
oracle = float(x.sum(dtype=np.float64))  # float64 oracle both rungs must match (exact-ish)
def reduce_serial(x):  # RUNG 0 (bad): serial accumulator (one dependency chain, no parallelism)
    s = 0.0  # single running total (loop-carried dependency: iteration k waits for k-1)
    for v in x:  # walk elements one by one in python
        s += float(v)  # accumulate (cannot vectorize across iterations)
    return s  # return the total
def reduce_tree(x, B=1024):  # RUNG 2 (good): two-level tree (block partials, then tiny final sum)
    parts = (x.reshape(-1, B).sum(axis=1, dtype=np.float64))  # phase 1: every block sums its chunk independently (parallel on gpu)
    return float(parts.sum())  # phase 2: sum the few partials (log-depth tree on gpu, one line here)
t_ser, s_ser = ms_once(lambda: reduce_serial(x))  # time the serial rung once (slow python loop)
assert abs(s_ser - oracle) / max(1.0, abs(oracle)) < 1e-4  # serial must match the oracle
print(f'serial python loop: {t_ser:.2f}s ({x.nbytes / t_ser / 1e9:.2f} GB/s)')  # serial line (bandwidth will look tragic)
t_np = time_op(lambda: x.sum())  # numpy vectorized sum timing
print(f'numpy sum:          {t_np*1e3:.2f}ms ({x.nbytes / t_np / 1e9:.1f} GB/s, {t_ser / t_np:.0f}x vs serial)')  # numpy line
t_tree = time_op(lambda: reduce_tree(x))  # tree-reduce timing
assert abs(reduce_tree(x) - oracle) / max(1.0, abs(oracle)) < 1e-5  # tree must match the oracle
print(f'tree reduce:        {t_tree*1e3:.2f}ms ({x.nbytes / t_tree / 1e9:.1f} GB/s, {t_ser / t_tree:.0f}x vs serial)')  # tree line
t_torch = time_op(lambda: xt.sum())  # torch sum timing (second engine, same pattern)
print(f'torch sum:          {t_torch*1e3:.2f}ms ({x.nbytes / t_torch / 1e9:.1f} GB/s)')  # torch line
print('cuda lesson: warp-shuffle tree reduce = log2(n) steps across 32 lanes, then one value per block to global mem (see p5 partials)')  # takeaway
if HAS_CUDA:  # gpu rung (dgx spark): torch gpu sum = vendor tree-reduction kernel
    xg = xt.to('cuda')  # move to gpu once (outside timer)
    tg = time_op(lambda: xg.sum())  # time the gpu tree reduction
    print(f'gpu torch sum:      {tg*1e3:.2f}ms ({x.nbytes / tg / 1e9:.1f} GB/s)')  # gpu line
    GPU_RED = tg  # stash for the csv
else:  # cpu-only box: record the skip
    print('gpu reduction rung SKIPPED (no cuda)')  # guidance
    GPU_RED = None  # marker for the csv writer
print('experiment C done: parallelism structure is performance (same sum, different depth)')  # section close

In [ ]:
opt_rows = [{'exp': 'A-coalesce', 'variant': 'numpy-contig', 'n': contig.size, 'time_s': t_good, 'metric': 'GB/s', 'val': gbs_good}, {'exp': 'A-coalesce', 'variant': 'numpy-strided', 'n': strided.size, 'time_s': t_bad, 'metric': 'GB/s', 'val': gbs_bad}, {'exp': 'A-coalesce', 'variant': 'torch-contig', 'n': contig.size, 'time_s': t_tgood, 'metric': 'GB/s', 'val': touched / t_tgood / 1e9}, {'exp': 'A-coalesce', 'variant': 'torch-strided', 'n': strided.size, 'time_s': t_tbad, 'metric': 'GB/s', 'val': touched / t_tbad / 1e9}]  # experiment-a csv rows
opt_rows.append({'exp': 'B-tiling', 'variant': 'naive-ijk-96', 'n': M, 'time_s': t_naive, 'metric': 'speedup-vs-naive', 'val': 1.0})  # naive baseline row
opt_rows.append({'exp': 'B-tiling', 'variant': 'blocked-py-96', 'n': M, 'time_s': t_blockpy, 'metric': 'speedup-vs-naive', 'val': t_naive / t_blockpy})  # tiled-python row (expect ~1x: no cliff)
for T, t in tiled.items():  # add one csv row per numpy-blocked tile size at realistic size
    opt_rows.append({'exp': 'B-tiling', 'variant': f'blocked-np-T{T}-384', 'n': M2, 'time_s': t, 'metric': 'GFLOP/s', 'val': 2 * M2 ** 3 / t / 1e9})  # tiled-structure row
opt_rows.append({'exp': 'B-tiling', 'variant': 'blas-384', 'n': M2, 'time_s': t_blas, 'metric': 'GFLOP/s', 'val': gflops_blas})  # blas ceiling row
for name, t, nbytes in [('serial', t_ser, x.nbytes), ('numpy', t_np, x.nbytes), ('tree', t_tree, x.nbytes), ('torch', t_torch, x.nbytes)]:  # experiment-c cpu rungs
    opt_rows.append({'exp': 'C-reduce', 'variant': name, 'n': N, 'time_s': t, 'metric': 'GB/s', 'val': nbytes / t / 1e9})  # one row per rung
if GPU_COAL is not None:  # gpu coalescing rows only when measured
    opt_rows.append({'exp': 'A-coalesce', 'variant': 'gpu-contig', 'n': contig.size, 'time_s': GPU_COAL[0], 'metric': 'GB/s', 'val': touched / GPU_COAL[0] / 1e9})  # gpu good row
    opt_rows.append({'exp': 'A-coalesce', 'variant': 'gpu-strided', 'n': strided.size, 'time_s': GPU_COAL[1], 'metric': 'GB/s', 'val': touched / GPU_COAL[1] / 1e9})  # gpu bad row
if GPU_RED is not None:  # gpu reduction row only when measured
    opt_rows.append({'exp': 'C-reduce', 'variant': 'gpu-torch', 'n': N, 'time_s': GPU_RED, 'metric': 'GB/s', 'val': x.nbytes / GPU_RED / 1e9})  # gpu rung row
with open('./results/opt.csv', 'w', newline='') as f:  # open optimization csv for writing
    w = csv.DictWriter(f, fieldnames=['exp', 'variant', 'n', 'time_s', 'metric', 'val'])  # writer with fixed columns
    w.writeheader()  # header first
    w.writerows(opt_rows)  # one row per (experiment, rung)
print('saved ./results/opt.csv with', len(opt_rows), 'rows')  # confirm csv path + size
for r in opt_rows:  # echo the full ladder table into the notebook
    print(f"{r['exp']:>10s} {r['variant']:>18s} time={r['time_s']*1e3:10.2f}ms {r['metric']}={r['val']:.2f}")  # one line per rung
fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # figure with two panels: bandwidth + speedups
a_rows = [r for r in opt_rows if r['exp'] == 'A-coalesce']  # filter experiment-a rows for the left panel
axes[0].bar([r['variant'] for r in a_rows], [r['val'] for r in a_rows])  # bar per rung (gb/s, higher better)
axes[0].set(title='Experiment A: achieved bandwidth by access pattern', ylabel='GB/s')  # left labels
axes[0].tick_params(axis='x', rotation=20)  # rotate labels so variants stay readable
c_rows = [r for r in opt_rows if r['exp'] == 'C-reduce']  # filter experiment-c rows for the right panel
axes[1].bar([r['variant'] for r in c_rows], [r['time_s'] * 1e3 for r in c_rows])  # bar per rung (ms, lower better)
axes[1].set_yscale('log')  # log y: serial dwarfs the vectorized rungs otherwise
axes[1].set(title='Experiment C: reduction time by rung (log ms, lower is better)', ylabel='ms (log)')  # right labels
axes[1].tick_params(axis='x', rotation=20)  # rotate labels for readability
fig.tight_layout()  # fix overlapping labels
fig.savefig('./results/opt.png', dpi=120)  # save the figure
plt.show()  # display inline
print('figure saved: ./results/opt.png')  # confirm plot path
print('what-if tile=8: tiles fit easily but the kk-loop overhead + short inner loops starve the pipeline (too-fine granularity)')  # what-if small tile
print('what-if tile=96: one tile = whole matrix (no reuse benefit, same as naive + loop overhead)')  # what-if huge tile
print('profiler guidance: ncu --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,l1tex__t_bytes.sum per kernel; nsys profile --trace=cuda,nvtx python train.py')  # exact next commands

In [ ]:
print('=== nvidia-smi snapshot (power/thermal/vram/util) ===')  # header for the tooling section
if HAS_SMI:  # driver tool present: take a real snapshot
    try:  # smi can fail (no gpu, driver hiccup): never kill the notebook
        q = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu', '--format=csv'], capture_output=True, text=True, timeout=30)  # one csv query
        print(q.stdout.strip())  # print the gpu table (1 line per gpu)
        with open('./results/smi.txt', 'w') as f:  # archive the snapshot next to the csv
            f.write(q.stdout)  # write raw smi output for the report
        print('archived ./results/smi.txt')  # confirm archive path
    except Exception as e:  # any smi failure degrades to a message
        print('nvidia-smi query failed:', str(e)[:150])  # log short reason
else:  # no driver tool on this box: print the dgx runbook instead
    print('nvidia-smi NOT found (cpu-only box). On DGX Spark run:')  # state the situation
    print('  nvidia-smi dmon -s pucvmet -d 1   # live power/util/clock/mem/thermal while the notebook benchmarks')  # live-monitor command
    print('  nvidia-smi --query-gpu=... --format=csv -l 1 > results/smi.csv  # log to csv alongside opt.csv')  # logging command
print('nsight runbook (DGX Spark): nsys profile -o rep --trace=cuda,nvtx jupyter nbconvert --execute notebook.ipynb; ncu --set detailed -o rep --kernel-name ...')  # profiler commands
print('what to read in ncu: memory-workload-analysis (dram vs l1 hit), sm-throughput (are sms fed?), launch-stats (is the grid big enough?)')  # reading guide
print('roofline wrap-up: A moved UP (more gb/s via coalescing); B moved RIGHT (fewer bytes via reuse); C moved DOWN in depth (log tree vs serial chain)')  # synthesis
print('cleanup: big/partial buffers are numpy/cpu tensors (freed on rerun); rerun on dgx spark to append gpu-* rows')  # next steps
print('next: session 3 spends these gb/s on convnets (P7: im2col vs winograd vs cudnn, same roofline game)')  # forward pointer

## What to write in your README (7 questions)
1. What did I build? 2. How do coalescing / tiling / tree-reduce work (one line each)? 3. Measured gaps: strided slowdown, best tile speedup, serial-vs-tree factor?
4. What surprised you (numpy strided gap on CPU? blas ceiling height?)? 5. Bottleneck per experiment: what resource starved (dram transactions, cache reuse, dependency chain)?
6. What if tile=8 / tile=whole-matrix / transfer-included? Why? 7. Next: what would ncu confirm, and which rung would you push further?
**One-liner:** same flops ≠ same speed — contiguous access feeds the machine, tiling reuses bytes, trees shorten dependency chains; profile (smi → nsys → ncu) to find which one starves first.
